# Extensible Event Format (EVE)

## Intro

* Suricata is not just IDS;
* Lot of protocol parsers;
* Need to parse a protocol in order to implement alerting;
* Logging the parsed data is a no brainer;
* EVE stands for Extensible Event Format;
* Suricata is NSM first, IDS second;
    * most of EVE is protocol metadata;
    * alerts are a thin layer on top;

### Starting to learn

* Full JSON logging of protocol metadata;
* More in [official documentation](https://docs.suricata.io/en/latest/output/eve/eve-json-output.html);
* Good resource to learn is [Malware Traffic Analysis (dot net)](https://www.malware-traffic-analysis.net/)

In [ ]:
import json

In [ ]:
from glob import glob
FILES = glob("./data/*/eve.json")

In [ ]:
EVE = []
for f in FILES:
    with open(f, "r") as handle:
        docs = [json.loads(l) for l in handle]
        docs = [{**d, "file": f} for d in docs]
        print(f"got {len(docs)} events from {f}")
        EVE += docs

### Legacy alerts

* `fast.log` - Snort heritage, one flat line per alert;
* Signature, classification, priority, 5-tuple;
* No protocol metadata, no flow context;
* Only alerts - everything else is invisible;

```
10/09/2026-20:49:16.061935  [**] [1:2061871:2] ET MALWARE AMOS ClickFix Campaign - Bash Stager M2 [**] [Classification: A Network Trojan was detected] [Priority: 1] {TCP} 10.10.9.220:49155 -> 45.147.31.197:80
10/09/2026-20:49:16.061935  [**] [1:2034567:1] ET HUNTING curl User-Agent to Dotted Quad [**] [Classification: Potentially Bad Traffic] [Priority: 2] {TCP} 10.10.9.220:49155 -> 45.147.31.197:80
10/09/2026-20:49:30.295815  [**] [1:2013790:4] ET POLICY External IP Lookup ip-api.com [**] [Classification: Device Retrieving External IP Address Detected] [Priority: 1] {TCP} 10.10.9.220:49158 -> 208.95.112.1:80
10/17/2026-00:26:25.259665  [**] [1:2400002:4185] ET DROP Spamhaus DROP Listed Traffic Inbound group 3 [**] [Classification: Misc Attack] [Priority: 2] {TCP} 192.0.2.77:51234 -> 10.10.9.220:443
08/12/2026-16:12:49.167142  [**] [1:2027716:2] ET INFO Possible Lateral Movement - File Creation Request in Remote System32 Directory (T1105) [**] [Classification: Misc activity] [Priority: 3] {TCP} 10.137.1.33:49757 -> 10.137.1.62:445
```

### Extensible Event Format

* One JSON object per line;
* Common header - timestamp, 5-tuple, `event_type`, ...;
* Then a section named after `event_type`;
* Few examples of different types;

In [ ]:
EXAMPLES = {}
for e in EVE:
    EXAMPLES.setdefault(e["event_type"], e)

In [ ]:
from IPython.display import JSON

In [ ]:
JSON(EXAMPLES["alert"])

### Into a table

* `json_normalize` flattens nested sections into dotted columns;
* `http.hostname`, `alert.signature`, `flow.bytes_toserver`, ...;

In [ ]:
import pandas as pd
DF = pd.json_normalize([e for e in EVE if e.get("event_type", "") not in ["stats"]])
DF["flow_id"] = DF["flow_id"].astype("Int64")

In [ ]:
len(DF)

In [ ]:
DF

## Core concepts

### Event type

* To reiterate **protocol log is produced regardless if flow alerted or not**
* `event_type` field refers to what kind of event we're dealing with
* Event types fall into groups:
    * security - `alert`, `anomaly`, `drop`;
    * application layer - `http`, `dns`, `tls`, `smb`, `dcerpc`, `krb5`, ...;
    * flow - `flow` (both directions in one record), `netflow` (one record per direction);
    * file - `fileinfo`;

In [ ]:
JSON(EXAMPLES["dns"])

In [ ]:
JSON(EXAMPLES["http"])

In [ ]:
JSON(EXAMPLES["tls"])

In [ ]:
JSON(EXAMPLES["flow"])

In [ ]:
(
    DF
    .groupby("event_type")
    .agg({"event_type": ["count"]})
    .sort_values(by=[("event_type", "count")], ascending=False)
)

### NSM vs IDS

* How much of EVE is actually alerts?
* Notice - more HTTP events than alerts;
* Even more flows!
* Many connections with no corresponding alerts;
* `flow` is logged for every connection, so look with and without it;

In [ ]:
(
    DF
    .assign(alert=DF.event_type == "alert", no_flow=DF.event_type != "flow")
    .pipe(lambda df: pd.Series({
        "alerts / all events": f"{(df.alert.sum() / len(df)) * 100}%",
        "alerts / non-flow events": f"{(df.alert.sum() / df.no_flow.sum()) * 100}%" ,
    }))
)

### Flow ID

* Multiple events emitted per flow;
* Each flow has unique `flow_id`;
* Some protocols emit more than one event;
* Unknown protocols will still emit `flow`;
* Single flow with a lot of events can be an indicator;
* Count connections, not events;

In [ ]:
DF_FLOW_ID = (
    DF
    .groupby("flow_id")
    .agg({"flow_id": ["count"]})
    .sort_values(by=[("flow_id", "count")], ascending=False)
)
DF_FLOW_ID

In [ ]:
pd.Series({"events": len(DF), "flows": DF.flow_id.nunique()})

### Flow correlation

* Flow correlation is the technique of tying together multiple events with same flow ID;
* Pivot hunting:
    * find interesting value;
    * extract `flow_id`;
    * query all events with that `flow_id`;
    * then rinse and repeat;
* Sometimes we can *kinda* get away without doing that;

In [ ]:
(
    DF
    .loc[DF["http.url"] == "/41a7b0ffa7748712?force=1"]
    .dropna(how="all", axis=1)
)

In [ ]:
FLOW_ID = (
    DF
    .loc[DF["http.url"] == "/41a7b0ffa7748712?force=1"]
    .flow_id
    .unique()
    [0]
)
int(FLOW_ID)

In [ ]:

(
    DF
    .loc[DF.flow_id == FLOW_ID]
    [[
        "timestamp", "event_type", "flow_id", "src_ip", "dest_ip", "dest_port",
        "alert.signature", "http.hostname", "http.url", "http.http_user_agent",
        "fileinfo.filename", "fileinfo.size", "flow.bytes_toserver", "flow.bytes_toclient", "flow.state",
    ]]
)

### Event timing

* Events are written at different moments:
    * `alert` - instantly, when a rule matches;
    * application layer - per parsed transaction;
    * `flow` - at the end, when connection closes or times out;
* `alert` carries a `flow` section, but it's a snapshot at match time;
    * only the `flow` event has final bytes, packets, duration and state;
* Connection still open means **no `flow` event yet**;
    * app layer events already visible, final byte counts missing;
* `flow` event `timestamp` is flow start, not when it was written;
    * file order is the write order;

### Alerts are just events

* `alert` is just another `event_type`;
* Same header - timestamp, `flow_id`, 5-tuple;
* `payload` and `payload_printable` expose what triggered the rule;
* Join to protocol events by `flow_id`;
* `severity` is pretty much useless;
* ruleset and category for fidelity estimation;
* alerts are directional!

In [ ]:
(
    DF
    .loc[~DF.app_proto.fillna("").eq("http")]
    .loc[DF.event_type == "alert"]
    [["timestamp", "community_id", "src_ip", "src_port", "dest_ip", "dest_port", "app_proto", "alert.signature", "alert.severity", "payload_printable"]]
)

### Hunting alerts

* Signatures expose what Suricata may not;
* Suricata is not wireshark - cannot parse everything;
* Malware often does not follow any RFC;
* Hunting signatures highlight interesting stuff, expose payload;
* These events should be analyzed like any other log record, not for escalation;
* **Noisy hunting alert != false positive!!!**

In [ ]:
(
    DF
    .loc[DF.community_id == "1:yvSJ9U8mV/FPfblZ2rh+H8IsFbc="]
    [["timestamp", "src_ip", "dest_ip", "dest_port", "event_type", "alert.signature", "smb.command", "smb.status", "payload_printable", "flow.pkts_toserver", "flow.pkts_toclient"]]
)

### High fidelity alerts

In [ ]:
pd.DataFrame(DF["alert.signature"].unique())

In [ ]:
DF.loc[DF["alert.signature"].fillna("").str.contains("^ET MALWARE")].dropna(how="all", axis=1)

### Alert tiers

* **high** - near certain, known bad, IoC hits, C2 signatures;
    * rare, standalone detection;
* **medium** - legit feature used suspiciously, remote admin, tool execution;
    * expected from this source at this time?
* **low** - noise in isolation, `ET INFO`, `ET POLICY`, hunting rules;
    * never dismiss;

## Core fields

### Common fields

* Every event shares a common header;
* `timestamp` - ISO 8601 with timezone;
* `flow_id` - join key across all events of one flow;
* `tx_id` - transaction order within a flow;
* `src_ip`, `src_port`, `dest_ip`, `dest_port`, `proto` - 5-tuple;
* `app_proto` - detected application protocol;
* `community_id` - 5-tuple hash, correlates with other tools (Zeek, Arkime, ...);
* Type specific fields sit under prefix matching `event_type`;

In [ ]:
(
    DF
    .drop_duplicates("event_type")
    [[
        "event_type", "timestamp", "flow_id", "tx_id", "community_id",
        "src_ip", "src_port", "dest_ip", "dest_port", "proto", "app_proto",
    ]]
)

### Cardinality

* How many distinct values a field holds;
* Picks the strategy:
    * low - enumerate unique values, anything unexpected is an anomaly;
    * medium - unique values with counts, outliers stand out;
    * high - drop frequent patterns, the tail is interesting;

In [ ]:
(
    DF[[
        "event_type", "proto", "app_proto", "http.http_method",
        "tls.sni", "smb.filename", "krb5.sname", "alert.signature", "http.hostname",
        "http.url", "http.http_user_agent", "flow_id",
    ]]
    .nunique()
    .sort_values()
)

### Key fields

* Handful of fields worth knowing;
* What did the connection *do*, not just what protocol it spoke;
* Protocol sections are not exclusive to their `event_type`:
    * `alert` carries metadata of the transaction that triggered it;
    * `fileinfo` carries metadata of the transaction that carried the file;

| Field | Appears in `event_type` |
|---|---|
| `http.hostname` | `http`, `alert`, `fileinfo` |
| `http.url` | `http`, `alert`, `fileinfo` |
| `http.http_method` | `http`, `alert`, `fileinfo` |
| `http.http_user_agent` | `http`, `alert`, `fileinfo` |
| `fileinfo.filename` | `fileinfo` |
| `fileinfo.size` | `fileinfo` |
| `tls.sni` | `tls` |
| `tls.ja3.hash` | `tls` |
| `tls.ja3s.hash` | `tls` |
| `tls.ja4` | `tls` |
| `tls.subject` | `tls` |
| `dns.queries` | `dns`, `alert` |
| `krb5.cname` | `krb5` |
| `krb5.sname` | `krb5` |
| `smb.command` | `smb`, `alert`, `fileinfo` |
| `smb.status` | `smb`, `alert`, `fileinfo` |
| `smb.share` | `smb`, `fileinfo` |
| `smb.filename` | `smb`, `fileinfo` |
| `dcerpc.interfaces` | `dcerpc` |
| `dcerpc.req.opnum` | `dcerpc` |
| `flow.bytes_toserver` / `flow.bytes_toclient` | `flow`, `alert` (partial) |
| `flow.state` | `flow` |
| `alert.signature` | `alert` |
| `payload` | `alert` |
| `payload_printable` | `alert` |

In [ ]:
pd.set_option('display.max_columns', None)

#### `http.hostname`

* **What** - `Host` header of the request;
* **Hunting** - should be FQDN, IP address or missing host means scripted tooling;
* **Properties** - client controlled, medium cardinality;

In [ ]:
(
    DF
    .loc[DF.event_type=="http"][
    [c for c in list(DF.columns.values) if c.startswith("http.")]]
    .dropna(how="all")
    ["http.hostname"]
    .unique()
)

#### `http.url`

* **What** - request path and query;
* **Hunting** - path traversal, command injection, probing for vulnerable paths;
* **Properties** - normalized without hostname, high cardinality;

In [ ]:
sorted(list((
    DF
    .loc[DF.event_type=="http"][
    [c for c in list(DF.columns.values) if c.startswith("http.")]]
    .dropna(how="all")
    ["http.url"]
    .dropna()
    .unique()
)))

#### `http.http_method`

* **What** - request verb;
* **Hunting** - well defined set, anything else is an indicator;
* **Properties** - low cardinality;

In [ ]:
sorted(list((
    DF
    .loc[DF.event_type=="http"][
    [c for c in list(DF.columns.values) if c.startswith("http.")]]
    .dropna(how="all")
    ["http.http_method"]
    .dropna()
    .unique()
)))

#### `http.http_user_agent`

* **What** - client software identifier;
* **Hunting** - tools ship default user-agents nobody bothers to change;
* **Properties** - client controlled, high cardinality, drop frequent patterns;

In [ ]:
DATA_UA = (
    DF
    .loc[DF.event_type=="http"][
    [c for c in list(DF.columns.values) if c.startswith("http.")]]
    .dropna(how="all")
    ["http.http_user_agent"]
    .dropna()
    .unique()
)
DATA_UA = list(DATA_UA)
DATA_UA = sorted(DATA_UA)
DATA_UA

In [ ]:
[d for d in DATA_UA if not d.startswith("Mozilla/5.0")]

#### `fileinfo.filename`

* **What** - name of transferred file - URL path for HTTP, file path for SMB;
* **Hunting** - what was actually delivered, not just requested;
* **Properties** - HTML and 404 pages are files too;

In [ ]:
sorted(list((
    DF
    .loc[DF.event_type == "fileinfo"]
    .dropna(how="all", axis=1)
    ["fileinfo.filename"]
    .dropna()
    .unique()
)))

#### `fileinfo.size`

* **What** - bytes transferred;
* **Hunting** - payload size, truncated or stored partially;
* **Properties** - numeric, high cardinality, hash and magic only if enabled in config;

In [ ]:
(
    DF
    .loc[DF.event_type == "fileinfo", ["fileinfo.filename", "fileinfo.size", "fileinfo.state"]]
    .sort_values(by="fileinfo.size", ascending=False)
    .head(20)
)

#### `tls.sni`

* **What** - server name requested in TLS handshake;
* **Hunting** - destination visible despite encryption, missing SNI means scripted tooling;
* **Properties** - client controlled, optional, medium cardinality;

In [ ]:
DF.loc[DF.event_type == "tls", "tls.sni"].value_counts(dropna=False)

#### `tls.ja3.hash`

* **What** - fingerprint of TLS client hello;
* **Hunting** - identifies client software regardless of destination;
* **Properties** - low cardinality, older standard;

In [ ]:
DF.loc[DF.event_type == "tls", "tls.ja3.hash"].value_counts()

#### `tls.ja3s.hash`

* **What** - fingerprint of TLS server hello;
* **Hunting** - JA3 and JA3S pair identifies client and server combination, e.g. C2 framework talking to its server;
* **Properties** - low cardinality, missing when handshake is not seen to the end;

In [ ]:
DF.loc[DF.event_type == "tls", "tls.ja3s.hash"].value_counts(dropna=False)

#### `tls.ja4`

* **What** - newer TLS client fingerprint, human readable prefix;
* **Hunting** - same as JA3, prefix tells TLS version, SNI presence, ALPN;
* **Properties** - low cardinality;

In [ ]:
DF.loc[DF.event_type == "tls", "tls.ja4"].value_counts()

#### `tls.subject`

* **What** - server certificate subject;
* **Hunting** - self signed and random looking names on C2 servers;
* **Properties** - only TLS 1.2 and older - TLS 1.3 encrypts the certificate;

In [ ]:
DF.loc[DF.event_type == "tls", "tls.subject"].value_counts(dropna=False)

#### `dns.queries`

* **What** - list of queried names and record types;
* **Hunting** - C2 and staging domains, IP lookup services, DGA;
* **Properties** - medium cardinality, separate request and response events;

In [ ]:
(
    pd.json_normalize(
        DF
        .loc[DF.event_type == "dns", "dns.queries"]
        .explode()
        .dropna()
        .tolist()
    )
    .value_counts()
)

#### `krb5.cname`

* **What** - client principal - who is asking;
* **Hunting** - which accounts authenticate from where;
* **Properties** - low cardinality, often empty in requests;

In [ ]:
DF.loc[DF.event_type == "krb5", "krb5.cname"].value_counts()

#### `krb5.sname`

* **What** - service principal - ticket for what;
* **Hunting** - which services an account reaches, unusual service tickets;
* **Properties** - medium cardinality;

In [ ]:
DF.loc[DF.event_type == "krb5", "krb5.sname"].value_counts()

#### `smb.command`

* **What** - SMB operation;
* **Hunting** - tree connect, create and write sequence is file drop;
* **Properties** - low cardinality, one event per command;

In [ ]:
DF.loc[DF.event_type == "smb", "smb.command"].value_counts()

#### `smb.status`

* **What** - NT status code of the server response;
* **Hunting** - `STATUS_ACCESS_DENIED` is failed access to admin shares or files, failures followed by success show credential trial;
* **Properties** - low cardinality, `STATUS_SUCCESS` dominates;

In [ ]:
DF.loc[DF.event_type == "smb", "smb.status"].value_counts(dropna=False)

#### `smb.share`

* **What** - share connected to;
* **Hunting** - `ADMIN$`, `C$` and `IPC$` are admin shares, lateral movement staple;
* **Properties** - low cardinality, only on tree connect;

In [ ]:
DF.loc[DF.event_type == "smb", "smb.share"].value_counts()

#### `smb.filename`

* **What** - file or named pipe opened on share;
* **Hunting** - dropped executables, named pipes like `svcctl` for remote service control;
* **Properties** - medium cardinality;

In [ ]:
DF.loc[DF.event_type == "smb", "smb.filename"].value_counts()

#### `dcerpc.interfaces`

* **What** - API called over RPC, by UUID;
* **Hunting** - service control, WMI, domain replication;
* **Properties** - opaque, alerts label them;

In [ ]:
(
    DF
    .loc[DF.event_type == "dcerpc", ["dcerpc.interfaces"]]
    .explode("dcerpc.interfaces")
    .dropna()
    ["dcerpc.interfaces"]
    .apply(lambda i: i["uuid"])
    .value_counts()
)

#### `dcerpc.req.opnum`

* **What** - function number within interface;
* **Hunting** - create service, start service, ...;
* **Properties** - opaque, meaningful only with interface;

In [ ]:
(
    DF
    .loc[DF.event_type == "dcerpc", ["dcerpc.request", "dcerpc.req.opnum"]]
    .value_counts(dropna=False)
)

#### `flow.bytes_toserver / flow.bytes_toclient`

* **What** - bytes sent each way;
* **Hunting** - upload versus download, exfiltration, beacon size;
* **Properties** - final only in `flow` event, partial snapshot in `alert`;

In [ ]:
DF.loc[DF.event_type == "flow", ["flow.bytes_toserver", "flow.bytes_toclient"]].describe()

#### `flow.state`

* **What** - how the connection ended;
* **Hunting** - `new` means no reply was seen, typical for scanning;
* **Properties** - low cardinality - new, established, closed;

In [ ]:
DF.loc[DF.event_type == "flow", "flow.state"].value_counts()

#### `alert.signature`

* **What** - name of the rule that matched;
* **Hunting** - label for what NSM recorded;
* **Properties** - medium cardinality, prefix hints ruleset category;

In [ ]:
DF.loc[DF.event_type == "alert", "alert.signature"].value_counts()

#### `payload`

* **What** - base64 of packet or stream data that triggered the rule;
* **Hunting** - exact bytes the rule matched, decode for binary protocols;
* **Properties** - base64, high cardinality, only if payload logging is enabled in config;

In [ ]:
import base64
(
    DF
    .loc[DF.event_type == "alert", ["alert.signature", "payload"]]
    .dropna()
    .head()
    .assign(decoded=lambda df: df.payload.apply(base64.b64decode))
)

#### `payload_printable`

* **What** - same payload, non-printable bytes replaced with `.`;
* **Hunting** - read HTTP requests, commands and shell strings directly;
* **Properties** - human readable, lossy, high cardinality;

In [ ]:
(
    DF
    .loc[DF.event_type == "alert", ["alert.signature", "payload_printable"]]
    .drop_duplicates()
    .head(20)
)

## Advanced hunting

### Flow pivoting

* One flow rarely tells the whole story;
* Take a value from one flow, find every flow holding it, pull those flows;
* Each hop can surface a new value - rinse and repeat;
* Using malware infection dataset - AMOS ClickFix;

In [ ]:
DF_MW = DF.loc[DF.file.str.contains("malware_infection")]
KEY = [
    "timestamp", "event_type", "flow_id", "src_ip", "dest_ip", "dest_port",
    "alert.signature", "http.http_method", "http.hostname", "http.url", "http.http_user_agent",
    "fileinfo.filename", "fileinfo.size", "flow.bytes_toserver", "flow.bytes_toclient",
]

#### Overview

* What is in the dataset;
* How many connections, which protocols;
* What fired, on how many flows;

In [ ]:
pd.Series({"events": len(DF_MW), "flows": DF_MW.flow_id.nunique()})

In [ ]:
DF_MW.groupby("app_proto", dropna=False).flow_id.nunique()

In [ ]:
(
    DF_MW
    .loc[DF_MW.event_type == "alert"]
    .groupby("alert.signature")
    .flow_id
    .nunique()
    .sort_values(ascending=False)
)

#### Flow view

* Events are too granular, one row per connection instead;
* Ordered by flow start;
* Who talked to whom, on what port, how much;
* Which alerts fired on that connection;
* Few distinctive protocol values - what the connection *did*;
* HTTP flows only - that's where the case is;

In [ ]:
def uniq(s):
    return ", ".join(s.dropna().astype(str).unique())

DF_MW_FLOWS = (
    DF_MW
    .loc[DF_MW.app_proto == "http"]
    .assign(
        flow_src=lambda df: df.src_ip.where(df.event_type == "flow"),
        flow_dest=lambda df: df.dest_ip.where(df.event_type == "flow"),
        flow_port=lambda df: df.dest_port.where(df.event_type == "flow"),
    )
    .groupby("flow_id")
    .agg(
        start=("flow.start", "first"),
        src=("flow_src", "first"),
        dest=("flow_dest", "first"),
        port=("flow_port", "first"),
        events=("event_type", "size"),
        bytes_toserver=("flow.bytes_toserver", "max"),
        bytes_toclient=("flow.bytes_toclient", "max"),
        alerts=("alert.signature", uniq),
        url=("http.url", "first"),
    )
    .sort_values(by="start")
)
DF_MW_FLOWS

#### Timeline

* Same request repeated is one step, not many;
* First and last seen, how many connections, total volume;
* Reads as the story of the infection;

In [ ]:
(
    DF_MW_FLOWS
    .groupby(["dest", "url"])
    .agg(
        first=("start", "min"),
        last=("start", "max"),
        flows=("events", "size"),
        bytes_toserver=("bytes_toserver", "sum"),
        bytes_toclient=("bytes_toclient", "sum"),
        alerts=("alerts", lambda s: ", ".join(sorted(set(a for v in s for a in v.split(", ") if a)))),
    )
    .sort_values(by="first")
)

#### Anchor flow

* Start from the first `ET MALWARE` hit - the stager download;
* Pivot candidates - client IP, `curl` user-agent to a bare IP;

In [ ]:
ANCHOR = DF_MW_FLOWS.loc[DF_MW_FLOWS.alerts.str.contains("ET MALWARE")].index[0]
DF_MW.loc[DF_MW.flow_id == ANCHOR, KEY]

#### Pivot on user-agent

* Same client, same `curl` user-agent, other flows;
* Second stage download, much bigger;
* Repeated `/api/t` to a new IP on a non-standard port;

In [ ]:
(
    DF_MW
    .loc[
        (DF_MW.event_type == "http")
        & (DF_MW.src_ip == "10.10.9.220")
        & (DF_MW["http.http_user_agent"] == "curl/8.7.1")
    ]
    .groupby("flow_id")
    .agg(
        dest_ip=("dest_ip", "first"),
        dest_port=("dest_port", "first"),
        url=("http.url", "first"),
        length=("http.length", "first"),
    )
)

#### Pivot on destination

* New value - `109.120.178.160`;
* Every flow to it, by user-agent;
* Malware switched clients - user-agent pivot alone missed most of them;

In [ ]:
(
    DF_MW
    .loc[(DF_MW.event_type == "http") & (DF_MW.dest_ip == "109.120.178.160")]
    .groupby(["http.http_user_agent", "http.url"])
    .flow_id
    .nunique()
)

#### Drill into the odd one

* Beacons hit `/api/t`, but `/api/credentials` is something else;
* Pull the first such flow;
* `fileinfo` both ways - upload is bigger than the response;

In [ ]:
ODD = DF_MW.loc[DF_MW["http.url"] == "/api/credentials"].flow_id.unique()[0]
DF_MW.loc[DF_MW.flow_id == ODD, KEY]

#### Infection timeline

| Time | Phase | Flows | NSM evidence | Alerts |
|---|---|---|---|---|
| 20:49:15 | Delivery - stager | 1 to `45.147.31.197:80` | `curl` GET `/41a7b0ffa7748712?force=1`, 4726 byte script | ET MALWARE AMOS ClickFix Bash Stager, curl to dotted quad |
| 20:49:16 | Delivery - second stage | 1 to `95.163.152.66:80` | `curl` GET `/d966d77e5c6e448?force=1`, 44 MB down, file logged truncated | ET MALWARE AMOS ClickFix Bash Stager, curl to dotted quad |
| 20:49:16 - 20:49:54 | C2 beaconing | 30 to `109.120.178.160:8133` | `/api/t`, `curl` first then `Go-http-client/1.1` | ET HUNTING curl, ET USER_AGENTS Go HTTP Client |
| 20:49:30 | Discovery | 1 to `ip-api.com` | geolocation lookup `/json/?fields=...` | ET POLICY External IP Lookup |
| 20:49:31 - 20:51:10 | Exfiltration | 19 to `109.120.178.160:8133` | POST `/api/credentials`, 171 KB uploaded | Go user-agent only - 5 Safari user-agent uploads with no alert |

* IoCs - `45.147.31.197`, `95.163.152.66`, `109.120.178.160:8133`, `?force=1` URL pattern;
* `ET MALWARE` alert covers delivery only, rest was found by pivoting on NSM;

### Correlation

#### Hierarchy

* **atomic event** - one protocol record;
* **flow** - one connection, all events share `flow_id`;
* **sequence** - ordered set of flows that make up a higher order event;

#### Sequence

* Lateral movement dataset - PsExec style service creation;
* Same flow view, one row per connection;

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
pd.options.display.html.use_mathjax = False

In [ ]:
SVCCTL = DF.loc[DF.file.str.contains("smb_lateral_svcctl")]
pd.Series({"events": len(SVCCTL), "flows": SVCCTL.flow_id.nunique()})

In [ ]:
FLOW_VIEW = (
    SVCCTL
    .assign(ntlm=lambda df: df["smb.ntlmssp.domain"] + "\\" + df["smb.ntlmssp.user"])
    .groupby("flow_id")
    .agg(
        start=("flow.start", "first"),
        pcap_cnt=("pcap_cnt", "min"),
        src=("src_ip", "first"),
        dest=("dest_ip", "first"),
        port=("dest_port", "first"),
        proto=("app_proto", "first"),
        events=("event_type", "size"),
        bytes_toserver=("flow.bytes_toserver", "max"),
        bytes_toclient=("flow.bytes_toclient", "max"),
        alerts=("alert.signature", uniq),
        user=("krb5.cname", lambda s: uniq(s[s != "<empty>"])),
        ntlm_user=("ntlm", uniq),
        share=("smb.share", lambda s: uniq(s[s != ""])),
        filename=("smb.filename", lambda s: uniq(s[s != ""])),
    )
    .assign(metadata=lambda df: df[["user", "ntlm_user", "share", "filename"]].apply(
        lambda r: "; ".join(f"{k}: {v}" for k, v in r.items() if v), axis=1))
    .drop(columns=["user", "ntlm_user", "share", "filename"])
    .sort_values(by=["start", "pcap_cnt"])
)
FLOW_VIEW

#### Kill chain

| Time | Phase | Connections | NSM evidence | Alerts |
|---|---|---|---|---|
| 16:10:16 | Compromised user logon | Kerberos, DCERPC, `10.137.1.33` - `10.137.1.62` | `lab.user` tickets, LSA and endpoint mapper calls | none |
| 16:10:58 | Probing the remote share | SMB to `10.137.1.62:445` | `LAB\lab.user` tree connect `c$` | ET INFO NTLM negotiate, challenge |
| 16:12:49 | Privilege escalation to admin | SMB to `10.137.1.62:445` | NTLM session setup as `DOMAIN\lab.admin` | none |
| 16:12:49 | Staging the payload over SMB | same SMB connection | `c$`, writes `Windows\System32\ReportingServicesService301.exe`, 810 KB up | ET INFO Possible Lateral Movement (T1105), ET POLICY executable transfer |
| 16:12:50 - 16:13:11 | Installing the payload via SVCCTL | DCERPC to `10.137.1.62:135`, SMB to `10.137.1.62:445` | endpoint mapper lookup, `DOMAIN\lab.admin` opens `svcctl` pipe | ET RPC DCERPC SVCCTL, Stamus MS-SCMR open, create, start, delete service |
| 16:13:11 | C2 callback (not present in data sample) | `10.137.1.62` to public IP  | - | - |

* `10.137.1.33` (CLEARXDR-WS1) moves laterally to `10.137.1.62`;
* Account switch `lab.user` to `lab.admin` over NTLM, `DOMAIN` placeholder in domain field - tooling;
* Service created, started and deleted - cleanup;
* No high tier alert;